# Export Orion-XAI to GGUF (for the Hugging Face Space)

Merges the LoRA adapter into Llama 3.2 3B Instruct, converts it to a 4-bit GGUF file (~2 GB),
and uploads it to `<your-hf-username>/orion-xai-gguf`. The Space downloads this file when it builds.

**Kaggle settings before you run:**
1. Accelerator: **None** is fine (CPU). Takes about 20–30 minutes.
2. Internet: **On**
3. Add-ons → Secrets → add `HF_TOKEN`: a Hugging Face token with **write** access
   (huggingface.co → Settings → Access Tokens → Create new token → type *Write*)

In [ ]:
!pip install -q -U transformers peft accelerate huggingface_hub sentencepiece protobuf

In [ ]:
import os, gc, shutil
os.environ["HF_HOME"] = "/tmp/hf"          # keep big downloads out of /kaggle/working (20 GB limit)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

login(UserSecretsClient().get_secret("HF_TOKEN"))
HF_USER  = whoami()["name"]
BASE     = "unsloth/Llama-3.2-3B-Instruct"   # same weights the adapter was trained on
ADAPTER  = "SnowyIbrahim/orion-xai-adapter"
OUT_REPO = f"{HF_USER}/orion-xai-gguf"
print("Will upload to:", OUT_REPO)
print(shutil.disk_usage("/tmp"))

## 1. Merge the adapter into the base model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok   = AutoTokenizer.from_pretrained(BASE)
base  = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
model = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
model.save_pretrained("/tmp/merged", safe_serialization=True)
tok.save_pretrained("/tmp/merged")
del model, base; gc.collect()
shutil.rmtree("/tmp/hf", ignore_errors=True)   # free the downloaded base weights
print(os.listdir("/tmp/merged"))

## 2. Convert to GGUF and quantise to 4-bit

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
!pip install -q /tmp/llama.cpp/gguf-py
!python /tmp/llama.cpp/convert_hf_to_gguf.py /tmp/merged --outtype f16 --outfile /tmp/orion-xai-f16.gguf
shutil.rmtree("/tmp/merged")

In [ ]:
!cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DLLAMA_CURL=OFF -DGGML_NATIVE=OFF > /dev/null
!cmake --build /tmp/llama.cpp/build --target llama-quantize -j 4 > /dev/null
!/tmp/llama.cpp/build/bin/llama-quantize /tmp/orion-xai-f16.gguf /tmp/orion-xai-q4_k_m.gguf Q4_K_M
os.remove("/tmp/orion-xai-f16.gguf")
print(round(os.path.getsize("/tmp/orion-xai-q4_k_m.gguf") / 1e9, 2), "GB")

## 3. Upload to the Hugging Face Hub

In [ ]:
from huggingface_hub import create_repo, upload_file

create_repo(OUT_REPO, repo_type="model", exist_ok=True)
upload_file(path_or_fileobj="/tmp/orion-xai-q4_k_m.gguf",
            path_in_repo="orion-xai-q4_k_m.gguf", repo_id=OUT_REPO)

card = f"""---
base_model: unsloth/Llama-3.2-3B-Instruct
tags: [gguf, llama, xai, cybersecurity]
license: llama3.2
---
# Orion-XAI (GGUF, Q4_K_M)

[SnowyIbrahim/orion-xai-adapter](https://huggingface.co/SnowyIbrahim/orion-xai-adapter) merged into
Llama 3.2 3B Instruct and quantised to 4-bit for CPU inference with Ollama or llama.cpp.

Trained on the **Alpaca** prompt format; see the Modelfile in
[the GitHub repo](https://github.com/mohamedibrahim26/explainable-ai/tree/main/deploy/hf-space).
"""
upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=OUT_REPO)
print(f"Done: https://huggingface.co/{OUT_REPO}")